In [12]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

torch.manual_seed(1)

В тех случаях, когда нужно включить в модель новый слой, который еще не поддерживает PyTorch, мы можем определить новый класс, производный от класса nn.Module. Это
особенно полезно при разработке нового слоя или настройке существующего.
Рассмотрим простой пример создания пользовательского слоя. Допустим, нам нужно
определить новый линейный слой, который вычисляет w(x + e) + b, где e представляет
собой случайную величину (шумовую переменную).

## Кастомный слой NoisyLinear

- Наследует `nn.Module`; веса `w` и `b` объявлены как `nn.Parameter` — они будут обновляться оптимизатором.
- Шум (нормальное распределение) добавляется к входу **только при training=True**, при инференсе слой работает как обычный линейный: шум — регуляризация.

In [13]:
class NoisyLinear(nn.Module):
    def __init__(self, 
                 input_size, 
                 output_size, 
                 noise_stddev=0.1):
        super().__init__()
        w = torch.Tensor(input_size, output_size)
        self.w = nn.Parameter(w)
        nn.init.xavier_uniform_(self.w) 
        b = torch.Tensor(output_size).fill_(0) 
        self.b = nn.Parameter(b)
        self.noise_stddev = noise_stddev

    def forward(self, x, training=False):
        if training:
            noise = torch.normal(0.0, self.noise_stddev, x.shape)
            x_new = torch.add(x, noise) 
        else:
            x_new = x
        return torch.add(torch.mm(x_new, self.w), self.b)

## Модель с кастомным слоем

`NoisyLinear` встраивается в обычный MLP: NoisyLinear(2→4) → ReLU → Linear(4→4) → ReLU → Linear(4→1) → Sigmoid. Метод `predict` порогово решает класс по `>= 0.5`.

In [14]:
class MyNoisyModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = NoisyLinear(2, 4, 0.07)
        self.al = nn.ReLU()
        self.l2 = nn.Linear(4, 4)
        self.a2 = nn.ReLU()
        self.l3 = nn.Linear(4, 1)
        self.аз = nn. Sigmoid () 

    def forward(self, x, training=False):
        х = self.ll(x, training)
        х = self.al(x)
        х = self.x2(x)
        х = self.a2(x)
        х = self.lЗ(x)
        х = self.aЗ(x)
        return х

    def predict(self, x):
        х = torch.tensor(x, dtype=torch.float32)
        pred = self.forward(x) [:, 0]
        return (pred>=0.5).float() 

## Сборка модели

Создаём экземпляр и проверяем структуру — все слои на месте, включая кастомный `NoisyLinear`.

In [15]:
model = MyNoisyModule()
model

MyNoisyModule(
  (l1): NoisyLinear()
  (al): ReLU()
  (l2): Linear(in_features=4, out_features=4, bias=True)
  (a2): ReLU()
  (l3): Linear(in_features=4, out_features=1, bias=True)
  (аз): Sigmoid()
)